# Day 15 — Weighted average / convex combination

**Milestone 2:** softmax produced weights (days 13–14); now we *use* them. Attention's output is a **weighted average of value vectors** — the last building block before the real thing.

## 1. Review (~5 min)

Recall **day 1** (Σ linearity) and **day 13** (softmax).

In [ ]:
import numpy as np
rng = np.random.default_rng(15)

**R1.** (day 1) $\sum_i (a\,x_i + b\,y_i)$.

In [ ]:
def r_combo_sum(a, b, x, y):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r1(fn):
    n = 6; a = rng.normal(); b = rng.normal(); x = rng.normal(size=n); y = rng.normal(size=n)
    assert np.allclose(fn(a, b, x, y), a*np.sum(x) + b*np.sum(y))
    print("✅ Review R1 passed")

check_r1(r_combo_sum)

**R2.** (day 13) softmax.

In [ ]:
def r_softmax(s):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_r2(fn):
    s = rng.normal(size=5); e = np.exp(s - s.max())
    assert np.allclose(np.asarray(fn(s)), e / e.sum())
    print("✅ Review R2 passed")

check_r2(r_softmax)

## 2. Lecture note (~10 min): blending vectors with weights

**The itch.** Softmax tells us *how much* to attend to each item. To produce an output we must **combine** the items' value vectors accordingly.

**The picture.** Given value vectors $v_1,\dots,v_n$ and weights $w_1,\dots,w_n$ that are non-negative and sum to 1, the **weighted average** lands somewhere *among* the $v_i$ — pulled toward whichever has the most weight.

**The formula** (a Σ — straight from day 1's linearity):
$$\text{out} = \sum_{i=1}^{n} w_i\,v_i.$$
Stacking the values as rows of $V$ (shape $(n, d)$), this is just $\text{out} = w^\top V$ (shape $(d,)$). A whole *batch* of weight rows $W$ (shape $(m, n)$) gives $WV$ (shape $(m, d)$) — one blended output per row.

**Knobs & effect.** The weights dial the behavior: **one-hot** weights → pick a single value (hard select); **uniform** weights → the plain mean; **softmax** weights → a soft blend dominated by the top matches. Because $w_i\ge 0$ and $\sum w_i = 1$, the output is a **convex combination** — every coordinate stays within the min/max of that coordinate across the values (it can't shoot outside the data).

**In the wild.** The attention output ($\text{weights}\cdot V$), exponential moving averages, ensemble/model averaging, and mixture models.

## 3. Exercises (~15 min)

### Exercise 1 — weighted average
Given weights `w` (shape `(n,)`) and values `V` (shape `(n, d)`), return $\sum_i w_i V_i$ (shape `(d,)`). (Checked vs a loop oracle and $w^\top V$.)

In [ ]:
def weighted_average(w, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e1(fn):
    for _ in range(4):
        n = int(rng.integers(2, 6)); d = int(rng.integers(2, 6))
        w = rng.random(n); w = w / w.sum(); V = rng.normal(size=(n, d))
        out = np.asarray(fn(w, V)); assert out.shape == (d,)
        oracle = sum(w[i] * V[i] for i in range(n))
        assert np.allclose(out, oracle) and np.allclose(out, w @ V)
    print("✅ Exercise 1 passed")

check_e1(weighted_average)

### Exercise 2 — the weights set the behavior (the effect)
Using your weighted average inside `mix`, the check probes the limits: a **one-hot** `w` selects one value row; a **uniform** `w` returns the plain mean; and the result stays within the values' coordinatewise range (convexity).

In [ ]:
def mix(w, V):
    # just return weighted_average(w, V)
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e2(fn):
    V = rng.normal(size=(4, 3))
    oh = np.array([0.0, 0.0, 1.0, 0.0])
    assert np.allclose(fn(oh, V), V[2])                 # one-hot -> select
    uni = np.ones(4) / 4
    assert np.allclose(fn(uni, V), V.mean(axis=0))      # uniform -> mean
    w = rng.random(4); w = w / w.sum(); out = fn(w, V)  # convex bound
    assert np.all(out >= V.min(axis=0) - 1e-9) and np.all(out <= V.max(axis=0) + 1e-9)
    print("✅ Exercise 2 passed")

check_e2(mix)

### Exercise 3 — a whole batch of blends
Given a weight matrix `W` (shape `(m, n)`, each row a distribution) and values `V` (shape `(n, d)`), return `(m, d)` where row `r` is $\sum_j W_{rj} V_j$. (This is exactly the final step of attention.)

In [ ]:
def batched_weighted_average(W, V):
    raise NotImplementedError("# YOUR CODE HERE")

In [ ]:
def check_e3(fn):
    m, n, d = 3, 4, 5
    W = rng.random((m, n)); W = W / W.sum(axis=1, keepdims=True); V = rng.normal(size=(n, d))
    out = np.asarray(fn(W, V)); assert out.shape == (m, d)
    oracle = np.stack([sum(W[r, j] * V[j] for j in range(n)) for r in range(m)])
    assert np.allclose(out, oracle)
    print("✅ Exercise 3 passed")

check_e3(batched_weighted_average)

## Solutions (collapsed — peek only if stuck)

`ref_`-prefixed answers. Running the next cell validates everything against the checks above.

In [ ]:
def ref_combo_sum(a, b, x, y):
    return a*np.sum(x) + b*np.sum(y)

def ref_softmax(s):
    e = np.exp(s - np.max(s)); return e / e.sum()

def ref_weighted_average(w, V):
    return w @ V

def ref_mix(w, V):
    return ref_weighted_average(w, V)

def ref_batched_weighted_average(W, V):
    return W @ V

check_r1(ref_combo_sum)
check_r2(ref_softmax)
check_e1(ref_weighted_average)
check_e2(ref_mix)
check_e3(ref_batched_weighted_average)
print("\nAll reference solutions pass. 🎉  See you on day 16!")